# 04 · Mamba 双向扫描块

线性递推状态建模 + 卷积分支混合；教学版串行扫描，工程实现可换官方 Mamba 核。

In [ ]:
import torch
import torch.nn as nn


def scan_1d(x, direction="forward"):
    B, C, L = x.shape
    decay = torch.sigmoid(torch.randn(B, C, device=x.device)) * 0.9 + 0.05
    h = torch.zeros(B, C, device=x.device)
    outs = []
    idx = range(L) if direction == "forward" else range(L - 1, -1, -1)
    for t in idx:
        h = decay * h + x[:, :, t]
        outs.append(h)
    if direction == "backward":
        outs = outs[::-1]
    return torch.stack(outs, dim=2)


class BiScanBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.proj_in = nn.Conv2d(dim, dim * 2, 1)
        self.conv = nn.Conv2d(dim, dim, 3, padding=1, groups=dim)
        self.proj_out = nn.Conv2d(dim, dim, 1)
        self.norm = nn.GroupNorm(1, dim)

    def forward(self, x):
        B, C, H, W = x.shape
        x1, gate = self.proj_in(x).chunk(2, dim=1)
        fwd = scan_1d(x1.view(B, C, H * W), "forward").view(B, C, H, W)
        bwd = scan_1d(x1.view(B, C, H * W), "backward").view(B, C, H, W)
        ssm = (fwd + bwd) / 2
        local = self.conv(x1)
        y = (ssm + local) * torch.sigmoid(gate)
        return self.norm(x + self.proj_out(y))


class MambaDehazeNet(nn.Module):
    def __init__(self, dim=32, n_blocks=4):
        super().__init__()
        self.stem = nn.Conv2d(3, dim, 3, padding=1)
        self.body = nn.Sequential(*[BiScanBlock(dim) for _ in range(n_blocks)])
        self.head = nn.Conv2d(dim, 3, 3, padding=1)

    def forward(self, x):
        return x + self.head(self.body(self.stem(x)))


In [ ]:
model = MambaDehazeNet(dim=32, n_blocks=2)
x = torch.randn(1, 3, 128, 128)   # 教学版串行扫描，先用小尺寸验证
y = model(x)
print("输入:", x.shape, "输出:", y.shape)
print("参数量: %.2f M" % (sum(p.numel() for p in model.parameters()) / 1e6))
